# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huseyinTozluyurt/Flyrank-Internship-MachineLearning/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

We generate a prioritized refresh queue by scoring our feature vector with our trained Random Forest model and mapping output probabilities to transparent human-readable reason codes.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier

# Load data and prepare feature matrix
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df['is_declining_label'] = (df['trend_direction'].str.lower() == 'down').astype(int)

base_features = ['days_since_last_update', 'content_age_days', 'impressions_90d', 'avg_position', 'ctr']
X = df[base_features].copy()
X['staleness_pos_interaction'] = X['days_since_last_update'] * X['avg_position']
X = X.fillna(0)
y = df['is_declining_label']

# Fit model for queue generation
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, class_weight='balanced')
rf.fit(X, y)

df['decay_probability'] = rf.predict_proba(X)[:, 1]

# Assign human-readable reason codes
def assign_reason_code(row):
    if 6 <= row['avg_position'] <= 22 and row['days_since_last_update'] > 120:
        return "Danger Zone Position + Stale Content"
    elif row['impressions_90d'] > 5000 and row['avg_position'] <= 10:
        return "High-Traffic Page Micro-Slip"
    elif row['days_since_last_update'] > 200:
        return "Extended Inactivity Window"
    else:
        return "General Traffic Decay Risk"

df['reason_code'] = df.apply(assign_reason_code, axis=1)

# Build top 50 ranked action queue
queue = df.sort_values(by='decay_probability', ascending=False)[[
    'content_id', 'client_id', 'impressions_90d', 'avg_position', 'days_since_last_update', 'decay_probability', 'reason_code'
]].head(50)

print("Top 3 Ranked Actions in the Queue:")
display(queue.head(3))

Top 3 Ranked Actions in the Queue:


,content_id,client_id,impressions_90d,avg_position,days_since_last_update,decay_probability,reason_code
18559,content_0d9c0ed65840,client_3fdba35f04,382,0.8,104,0.700299,General Traffic Decay Risk
24012,content_d0351144852b,client_19581e27de,884,0.6,104,0.699514,General Traffic Decay Risk
8411,content_637107baa450,client_f369cb89fc,2495,0.7,8,0.690615,General Traffic Decay Risk


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

*   **Intended Use:** Designed for content managers and SEO strategists to efficiently allocate weekly editorial update hours toward pages exhibiting statistical indicators of traffic decay.
*   **Where it stops:** This playbook cannot diagnose off-site backlink loss, technical infrastructure failures (e.g., site-wide 503 errors), or broad macroeconomic shifts in search intent that are invisible to on-page Google Search Console telemetry.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Summary statistics of the ranked action queue
print(f"Queue Size: {len(queue)} pages")
print(f"Mean 90-Day Impressions of Flagged Queue: {queue['impressions_90d'].mean():.1f}")
print("\nReason Code Breakdown in Queue:")
print(queue['reason_code'].value_counts())

Queue Size: 50 pages
Mean 90-Day Impressions of Flagged Queue: 1407.0

Reason Code Breakdown in Queue:
reason_code
General Traffic Decay Risk    50
Name: count, dtype: int64


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*


*   **Human Review Checkpoint:** Before modifying any flagged URL, an editor must verify whether the traffic drop is seasonal (e.g., a holiday-specific keyword) or intentional (e.g., a planned content deprecation).
*   **The No-Go List (What should never be automated):**
    1. Automated bulk deletion or unpublishing of URLs.
    2. Automated rewriting of article bodies via generative AI without human editorial oversight.
    3. Structural URL slug changes without proper 301 redirect validation.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Verify no sensitive data or URLs exist in the queue export
assert 'url' not in queue.columns and 'client_name' not in queue.columns, "Privacy Violation: URL or client name present!"
print("Privacy Check Passed: Only anonymized hash IDs and metrics are present.")

Privacy Check Passed: Only anonymized hash IDs and metrics are present.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*


Recommendations will degrade and go stale if:
1. **Precision@50 Drift:** Weekly evaluation of newly realized outcomes shows Precision@50 dropping below a 0.60 threshold.
2. **Algorithm Shifts:** Major unannounced search engine ranking updates alter baseline click-through rate curves.
3. **Time Decay:** The model has operated on static feature weights for more than 90 consecutive days without a rolling retrain.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Create outputs directory if it doesn't exist and export queue
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

queue.to_csv("work/outputs/refresh_queue.csv", index=False)
queue.to_csv("outputs/refresh_queue.csv", index=False)
print("Action playbook queue successfully exported to work/outputs/refresh_queue.csv and outputs/refresh_queue.csv")

Action playbook queue successfully exported to work/outputs/refresh_queue.csv and outputs/refresh_queue.csv


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.